# RhymeAI — Gemma 4 LoRA fine-tune (Colab free-tier T4)

Trains **both** on-device adapters for RhymeAI (Writers Block) in one session:

| Run | Base model | Method | Output |
|---|---|---|---|
| 1 | `unsloth/gemma-4-E2B-it` | QLoRA (4-bit base) | `adapters/e2b/` |
| 2 | `unsloth/gemma-4-E4B-it` | QLoRA (4-bit base) | `adapters/e4b/` |
| 3 (optional bake-off) | `Qwen/Qwen3.5-4B` | QLoRA (4-bit base) | `adapters/qwen4b/` |

This notebook is the Colab port of `finetune/train/train_lora.py` (see
`finetune/train/RUNBOOK.md`). All of its logic is inlined below — same
hyperparameters (r=16/alpha=16, 2 epochs, lr 2e-4 cosine, warmup 3%,
adamw_8bit, effective batch 16, no packing, prompt-masked loss).

**Key design fact (do not change):** the dataset's `"text"` field is already
the app's exact raw turn template (Gemma `<start_of_turn>` form by default;
Qwen ChatML/no-think form for splits built with `prepare_dataset.py
--template qwen`). We do **not** apply the tokenizer chat template and do
**not** rebuild prompts here — train must match deploy byte-for-byte.

**Runtime:** the RUNBOOK's 4090 estimate is ~10–25 min per model on a ~1.4k-row
dataset. A free T4 is several times slower and the current dataset is ~5k rows,
so budget **roughly 1–3 hours total** for both models (30–60 min total only if
your dataset is small). Colab free tier can disconnect — every cell below is
re-run safe (upload and training are skipped if their outputs already exist).

> **Before running anything:** `Runtime → Change runtime type → T4 GPU`.


## 1. Install Unsloth

Same as RUNBOOK step 2: `pip install -U unsloth` pulls compatible
torch / transformers / trl / peft / datasets / bitsandbytes. No extra pins —
the training script does not pin either. Re-run safe (pip is idempotent).


In [ ]:
%pip install -U -q unsloth
import unsloth
print("unsloth", unsloth.__version__)


## 2. GPU check

Confirm a CUDA GPU is attached. On a T4 `bf16 supported` prints `False` —
the trainer config below mirrors `train_lora.py`'s fallback and will use
**fp16** mixed precision automatically (T4 has no bf16 hardware).


In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), (
    "No CUDA GPU. Select Runtime -> Change runtime type -> T4 GPU, then re-run.")
print("device      :", torch.cuda.get_device_name(0))
print("VRAM (GB)   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("bf16 support:", torch.cuda.is_bf16_supported(), "(False on T4 -> trainer uses fp16)")


## 2b. Mount Google Drive (adapter backup)

Adapters are multi-hundred-MB artifacts produced hours into the session;
backing each zip up to Drive the moment training finishes means a browser
or VM loss can never cost a finished run. Approve the OAuth prompt once.


In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/rhymeai-adapters"
os.makedirs(DRIVE_DIR, exist_ok=True)
print("adapter zips will be backed up to", DRIVE_DIR)


## 3. Upload the dataset

Upload **`train.jsonl`** and **`val.jsonl`** from your repo's
`finetune/train/out/` directory (produced by `prepare_dataset.py`; each row
already carries the full SFT `"text"` field).

Re-run safe: if both files are already on the Colab disk the upload prompt is
skipped.


In [ ]:
import json
import os

NEEDED = ["train.jsonl", "val.jsonl"]

missing = [f for f in NEEDED if not os.path.exists(f)]
if missing:
    print("Choose these file(s) from your repo's finetune/train/out/ :", ", ".join(missing))
    from google.colab import files
    files.upload()
else:
    print("train.jsonl and val.jsonl already present - skipping upload.")

missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"Still missing {missing} - re-run this cell and upload them."

_first_train_text = None
for name in NEEDED:
    n_rows = 0
    with open(name, encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            assert "text" in row, f"{name}: row {n_rows} has no 'text' field"
            if name == "train.jsonl" and n_rows == 0:
                _first_train_text = row["text"]
            n_rows += 1
    assert n_rows > 0, f"{name} is empty"
    print(f"{name}: {n_rows} rows")

print()
print("--- train.jsonl row 0, first 300 chars of 'text' ---")
print(_first_train_text[:300])


## 4. Shared training function

Inlined from `finetune/train/train_lora.py` — identical model load, PEFT
config, marker sanity check, SFT config, prompt masking, eval, adapter save.
T4-specific differences (see the intro): `load_in_4bit=True` for **both**
models and per-device batch 2 x grad-accum 8 (the script's own guidance for
4-bit on small cards; effective batch stays 16, so the LR schedule is
unchanged).

Carries the four post-v1 fixes from `train_lora.py`: fp32 dtype on bf16-less
GPUs (bnb dequant dtype mismatch), turn-marker gate demoted to a warning
(gemma-4 markers are plain text) with the multimodal-processor unwrap,
`fp16=False` always (gemma-4 upcasts to fp32 in unsloth), and no post-training
`evaluate()` (notebook progress-callback crash).

**After a disconnect, re-run this cell first** (it defines `train_one`), then
resume from whichever training cell you were on.


In [ ]:
import gc
import json
import os

# Unsloth must be imported BEFORE transformers/trl so its patches apply.
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only

import torch
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# Loss-mask split markers and tokenizer sanity probes per template. Must match
# prepare_dataset.py --template / eval/prompts.py TEMPLATES.
TEMPLATE_MARKERS = {
    "gemma": {
        "instruction": "<start_of_turn>user\n",
        "response": "<start_of_turn>model\n",
        "sanity": ("<start_of_turn>", "<end_of_turn>"),
    },
    # The qwen response marker is the tail of the empty-think prefill in the
    # prompt half — everything after it (completion + <|im_end|>) is supervised.
    "qwen": {
        "instruction": "<|im_start|>user\n",
        "response": "</think>\n\n",
        "sanity": ("<|im_start|>", "<|im_end|>"),
    },
}


def train_one(
    base,                    # unsloth/gemma-4-E2B-it or unsloth/gemma-4-E4B-it
    out_dir,                 # adapter -> <out_dir>/adapter, metrics -> <out_dir>/metrics.json
    load_in_4bit=True,       # QLoRA; True for both models on a 16 GB T4
    data="train.jsonl",
    val="val.jsonl",
    max_seq_length=2048,     # app runtime context (LocalModelService ctx)
    r=16,                    # per Unsloth Gemma 4 guide; drop to 8 if val loss overfits
    alpha=16,
    epochs=2.0,
    lr=2e-4,                 # docs default; short run so 2e-4 cosine is right
    batch_size=2,            # script default is 8; "lower to 2-4 with 4-bit on small cards"
    grad_accum=8,            # keeps effective batch 16, same as the script's 8x2
    seed=3407,
    template="gemma",        # gemma | qwen - must match prepare_dataset.py --template
):
    adapter_dir = os.path.join(out_dir, "adapter")
    if os.path.exists(os.path.join(adapter_dir, "adapter_config.json")):
        print(f"SKIP: {adapter_dir} already exists (delete it to retrain).")
        with open(os.path.join(out_dir, "metrics.json"), encoding="utf-8") as handle:
            return json.load(handle)["eval"]

    os.makedirs(out_dir, exist_ok=True)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model, tokenizer = FastModel.from_pretrained(
        model_name=base,
        max_seq_length=max_seq_length,
        # On bf16-less cards (T4) unsloth trains gemma-4 in fp32; the 4-bit
        # quantizer must dequantize to the SAME dtype or F.linear throws
        # 'float != c10::Half'. bf16 cards can pass None (auto).
        dtype=None if torch.cuda.is_bf16_supported() else torch.float32,
        load_in_4bit=load_in_4bit,
        full_finetuning=False,
    )

    markers = TEMPLATE_MARKERS[template]
    # Sanity: check how the deployed turn markers tokenize. gemma-4 treats
    # them as PLAIN TEXT (7 tokens each); Qwen's <|im_start|>/<|im_end|> are
    # single special tokens. Both are fine: the app sends the raw-string
    # template and stops on the eot STRING, and SFT teaches the model this
    # exact deployed surface form.
    for marker in markers["sanity"]:
        # FastModel may return a multimodal Processor; positional text lands
        # in its `images` arg. Use the inner tokenizer directly.
        ids = getattr(tokenizer, "tokenizer", tokenizer)(marker, add_special_tokens=False)["input_ids"]
        print("turn marker %r -> %d token(s) in %s" % (marker, len(ids), base))

    # Text-only LoRA on attention + MLP of the language tower. Vision stays
    # frozen; audio modules are never selected because we only opt into
    # language layers. Required for the GGUF export detour (unsloth #6808).
    model = FastModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=r,
        lora_alpha=alpha,
        lora_dropout=0,
        bias="none",
        random_state=seed,
    )
    model.print_trainable_parameters()

    dataset = load_dataset("json", data_files={"train": data, "val": val})

    bf16 = torch.cuda.is_bf16_supported()   # False on T4 -> fp16
    config = SFTConfig(
        output_dir=os.path.join(out_dir, "checkpoints"),
        dataset_text_field="text",
        max_length=max_seq_length,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="adamw_8bit",
        weight_decay=0.001,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        seed=seed,
        bf16=bf16,
        fp16=False,  # never fp16: gemma-4 upcasts to fp32 in unsloth; fp16 autocast causes dtype mismatch
        # No packing: examples are short, and packing would interleave
        # documents under the prompt-masking pass below.
        packing=False,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["val"],
        args=config,
    )

    # Mask everything before (and including) the template's response marker
    # so the loss covers only the completion + eot. Markers are the app's raw
    # turn markers (NOT the chat-template ones - we never applied
    # get_chat_template).
    trainer = train_on_responses_only(
        trainer,
        instruction_part=markers["instruction"],
        response_part=markers["response"],
    )
    # Show what the loss actually sees for row 0 - should be the completion
    # plus <end_of_turn>, nothing from the prompt.
    sample = trainer.train_dataset[0]
    supervised = [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100]
    print("--- supervised tokens of first row ---")
    print(tokenizer.decode(supervised))
    print("---------------------------------------")

    stats = trainer.train()
    print("train metrics:", stats.metrics)

    # Skip a final explicit evaluate(): per-epoch evals already ran, and in
    # notebook environments transformers' progress callback crashes with
    # 'on_train_begin must be called before on_evaluate' after train().
    eval_metrics = dict(stats.metrics or {})
    print("final eval metrics:", eval_metrics)

    peak_gb = round(torch.cuda.max_memory_reserved() / 1e9, 2)
    print(f"peak VRAM reserved: {peak_gb} GB")

    with open(os.path.join(out_dir, "metrics.json"), "w", encoding="utf-8") as handle:
        json.dump({
            "train": stats.metrics,
            "eval": eval_metrics,
            "peak_vram_gb": peak_gb,
            "args": {"base": base, "out": out_dir, "load_in_4bit": load_in_4bit,
                     "max_seq_length": max_seq_length, "r": r, "alpha": alpha,
                     "epochs": epochs, "lr": lr, "batch_size": batch_size,
                     "grad_accum": grad_accum, "seed": seed},
        }, handle, indent=2)

    # LoRA adapter - the artifact export_gguf.sh consumes.
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print("saved LoRA adapter ->", adapter_dir)

    # Free VRAM for the next run.
    del trainer, model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return eval_metrics


print("train_one() defined.")


## 5. Train run 1 — Gemma 4 E2B

RUNBOOK trains E2B as a plain bf16 LoRA on a 4090/A100 (~8–10 GB VRAM at
batch 8). A T4 has 16 GB but **no bf16** and less headroom, so here E2B also
loads in 4-bit (QLoRA) — safe and well within memory.

Loss sanity: Gemma 4 E2B/E4B **starting loss of 13–15 is normal** (documented
Unsloth behavior) — watch the trend and the per-epoch eval loss, not the
absolute number. If epoch 2 eval loss rises, retrain with `epochs=1.0` or
`r=8, alpha=16` (delete `adapters/e2b/adapter` first to un-skip).

Re-run safe: skipped automatically if `adapters/e2b/adapter` already exists.


In [ ]:
e2b_eval = train_one(
    base="unsloth/gemma-4-E2B-it",
    out_dir="adapters/e2b",
    load_in_4bit=True,
)
print("E2B final eval:", e2b_eval)


## 6. Free memory, then train run 2 — Gemma 4 E4B (QLoRA)

`train_one` already deletes its model/trainer and empties the CUDA cache on
exit, but this belt-and-braces cell clears anything lingering in the notebook
namespace before loading the second (larger) model.


In [ ]:
import gc
import torch

for _name in ("model", "trainer", "tokenizer"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"CUDA memory allocated after cleanup: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


E4B is QLoRA (`load_in_4bit=True`) exactly as in the RUNBOOK — bf16 LoRA on
E4B needs ~17 GB+ before activations, so 4-bit is mandatory on a 16 GB T4.

Re-run safe: skipped automatically if `adapters/e4b/adapter` already exists.


In [ ]:
e4b_eval = train_one(
    base="unsloth/gemma-4-E4B-it",
    out_dir="adapters/e4b",
    load_in_4bit=True,
)
print("E4B final eval:", e4b_eval)


In [ ]:
import os
import shutil

ckpt = "adapters/e4b/checkpoints"
if os.path.exists(ckpt):
    shutil.rmtree(ckpt)
zip_path = shutil.make_archive("rhymeai-e4b-v3-adapter", "zip", ".", "adapters/e4b")
print("zipped ->", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if "DRIVE_DIR" in globals():
    shutil.copy2(zip_path, DRIVE_DIR)
    print("backed up ->", DRIVE_DIR)
from google.colab import files
files.download(zip_path)


## 6b. Optional run 3 — Qwen3.5-4B (base-model bake-off)

Trains the same RhymeAI LoRA on `Qwen/Qwen3.5-4B` with the ChatML/no-think
template, using the splits from `prepare_dataset.py --template qwen`.
Self-contained: a fresh session only needs the install, GPU-check, and
`train_one` cells above before this section. Upload `train-qwen.jsonl` and
`val-qwen.jsonl` from your repo's `finetune/train/out/qwen/`.

Re-run safe: skipped automatically if `adapters/qwen4b/adapter` already exists.


In [ ]:
import os

NEEDED_QWEN = ["train-qwen.jsonl", "val-qwen.jsonl"]
missing = [f for f in NEEDED_QWEN if not os.path.exists(f)]
if missing:
    print("Upload from your repo's finetune/train/out/qwen/ :", ", ".join(missing))
    from google.colab import files
    files.upload()
missing = [f for f in NEEDED_QWEN if not os.path.exists(f)]
assert not missing, f"Still missing {missing} - re-run this cell and upload them."


In [ ]:
qwen_eval = train_one(
    base="Qwen/Qwen3.5-4B",
    out_dir="adapters/qwen4b",
    load_in_4bit=True,
    data="train-qwen.jsonl",
    val="val-qwen.jsonl",
    template="qwen",
)
print("Qwen3.5-4B final eval:", qwen_eval)


In [ ]:
import os
import shutil

ckpt = "adapters/qwen4b/checkpoints"
if os.path.exists(ckpt):
    shutil.rmtree(ckpt)
zip_path = shutil.make_archive("rhymeai-qwen4b-adapter", "zip", ".", "adapters/qwen4b")
print("zipped ->", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")

from google.colab import files
files.download(zip_path)


## 6c. Optional run 4 — Gemma 4 12B (Studio tier, v3 dataset)

Trains the 12B Studio-tier adapter (16GB+ devices: iPad Pro 1TB/2TB, M4
iPads, Macs, flagship phones with 12GB+) on the v3 dataset — completions
authored by STOCK 12B as teacher, never by a weaker model, to avoid
distilling craft downward. Standard Gemma template; the E4B/E2B v3 retrains
use the regular cells above with the same v3 splits.

Self-contained: a fresh session only needs the install, GPU-check, and
`train_one` cells above. Upload `train.jsonl` / `val.jsonl` produced by
`prepare_dataset.py --in finetune/data/out/train-v3.jsonl` (the shared
upload cell in section 3 handles them).

A 16 GB T4 is TIGHT for 12B QLoRA (~7 GB 4-bit weights + activations):
prefer an L4/A100 runtime; on a T4 drop to `batch_size=1, grad_accum=16`.

Re-run safe: skipped automatically if `adapters/g12b/adapter` already exists.


In [ ]:
g12b_eval = train_one(
    base="unsloth/gemma-4-12b-it",
    out_dir="adapters/g12b",
    load_in_4bit=True,
)
print("Gemma 12B final eval:", g12b_eval)


In [ ]:
import os
import shutil

ckpt = "adapters/g12b/checkpoints"
if os.path.exists(ckpt):
    shutil.rmtree(ckpt)
zip_path = shutil.make_archive("rhymeai-g12b-adapter", "zip", ".", "adapters/g12b")
print("zipped ->", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if "DRIVE_DIR" in globals():
    shutil.copy2(zip_path, DRIVE_DIR)
    print("backed up ->", DRIVE_DIR)

from google.colab import files
files.download(zip_path)


## 7. Package and download

Zips both adapter directories plus each run's `metrics.json` into
`rhymeai-gemma-adapters.zip` and downloads it. Reads final val losses from the
`metrics.json` files on disk, so it works even after a kernel restart.


In [ ]:
import json
import os
import shutil

for tag in ("e2b", "e4b"):
    metrics_path = f"adapters/{tag}/metrics.json"
    adapter_cfg = f"adapters/{tag}/adapter/adapter_config.json"
    assert os.path.exists(metrics_path), f"missing {metrics_path} - did run {tag} finish?"
    assert os.path.exists(adapter_cfg), f"missing {adapter_cfg} - did run {tag} finish?"
    with open(metrics_path, encoding="utf-8") as handle:
        metrics = json.load(handle)
    print(f"{tag.upper()}: final eval_loss = {metrics['eval'].get('eval_loss')} "
          f"(peak VRAM {metrics.get('peak_vram_gb', '?')} GB)")

# Drop heavy intermediate checkpoints from the zip - only the adapters and
# metrics are needed downstream (RUNBOOK step 5).
for tag in ("e2b", "e4b"):
    ckpt = f"adapters/{tag}/checkpoints"
    if os.path.exists(ckpt):
        shutil.rmtree(ckpt)

zip_path = shutil.make_archive("rhymeai-gemma-adapters", "zip", ".", "adapters")
print("zipped ->", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")

from google.colab import files
files.download(zip_path)


## 8. Next steps (back on the Mac)

1. Hand `rhymeai-gemma-adapters.zip` back to the Claude session (or unzip it
   yourself into the repo):
   - `adapters/e2b/adapter` -> `finetune/train/out/e2b-lora-v2-adapter`
   - `adapters/e4b/adapter` -> `finetune/train/out/e4b-qlora-v2-adapter`
   - the two `metrics.json` files -> `finetune/train/out/{e2b,e4b}-metrics.json`
2. Export GGUF locally with `finetune/train/export_gguf.sh` (RUNBOOK step 6).
   For the Qwen bake-off adapter use `--size QWEN4B --base-gguf
   ~/models/Qwen3.5-4B-BF16.gguf` (`hf download unsloth/Qwen3.5-4B-GGUF
   Qwen3.5-4B-BF16.gguf --local-dir ~/models`).
   Never use `save_pretrained_gguf` — that is the unsloth #6808 failure path.
3. Re-score with `finetune/eval/run_eval.py` (RUNBOOK step 7) and compare
   against `finetune/tuned-E2B.json` / `tuned-E4B.json` (the v1 scorecard).
   Ship only if format-pass improves without regressing rhyme misses.

Then close this Colab tab — free-tier GPU time is limited.
